# Python Mutation Testing (SuT: `calculate_shipping_fee`)

In this exercise, we explore **mutation testing** using [mutmut](https://mutmut.readthedocs.io/), a mutation testing tool for Python.

Mutation testing evaluates the quality of your test suite by introducing small changes (mutations) to your code and checking whether your tests detect them. A **killed mutant** means your tests caught the change; a **survived mutant** indicates a gap in your test coverage.

We'll use the `calculate_shipping_fee` function as our System under Test (SuT) and learn how to:
1. Run mutation testing with `mutmut`
2. Interpret mutation testing results
3. Improve test suites to kill surviving mutants

Run this notebook with the working directory set to `modules/03_mutation_testing/exercises/` (so that `tests_shipping_fee/` is on the expected path). Use **`--tests-dir tests_shipping_fee`** with `mutmut run` in this layout (no top-level `tests/` folder) to avoid mutmut 2.x `tests_dir` issues.

## Check how to use `mutmut`

In [21]:
!pytest --version
!mutmut version

pytest 8.3.3
mutmut version 2.4.4


In [22]:
!pytest --cov=calculate_shipping_fee --cov-branch tests_shipping_fee

============================= test session starts ==============================
platform linux -- Python 3.10.20, pytest-8.3.3, pluggy-1.6.0
rootdir: /workspace/modules/03_mutation_testing/exercises
plugins: anyio-4.13.0, cov-5.0.0
collected 3 items                                                              

tests_shipping_fee/test_calculate_shipping_fee.py ...                    [100%]

---------- coverage: platform linux, python 3.10.20-final-0 ----------
Name                                           Stmts   Miss Branch BrPart  Cover
--------------------------------------------------------------------------------
tests_shipping_fee/calculate_shipping_fee.py      22      3     16      3    84%
--------------------------------------------------------------------------------
TOTAL                                             22      3     16      3    84%


============================== 3 passed in 0.01s ===============================


In [23]:
!rm -rf .mutmut-cache

In [24]:
!mutmut -h

Usage: mutmut [OPTIONS] COMMAND [ARGS]...

  Mutation testing system for Python.

Options:
  -h, --help  Show this message and exit.

Commands:
  apply       Apply a mutation on disk.
  html        Generate a HTML report of surviving mutants.
  junitxml    Show a mutation diff with junitxml format.
  result-ids  Print the IDs of the specified mutant classes (separated by...
  results     Print the results.
  run         Runs mutmut.
  show        Show a mutation diff.
  version     Show the version and exit.


In [25]:
!mutmut run -h

Usage: mutmut run [OPTIONS] [ARGUMENT]

  Runs mutmut. You probably want to start with just trying this. If you supply
  a mutation ID mutmut will check just this mutant.

Options:
  --paths-to-mutate TEXT
  --disable-mutation-types TEXT   Skip the given types of mutations.
  --enable-mutation-types TEXT    Only perform given types of mutations.
  --paths-to-exclude TEXT
  --runner TEXT
  --use-coverage
  --use-patch-file TEXT           Only mutate lines added/changed in the given
                                  patch file
  --rerun-all                     If you modified the test_command in the
                                  pre_mutation hook, the default test_command
                                  (specified by the "runner" option) will be
                                  executed if the mutant survives with your
                                  modified test_command.
  --tests-dir TEXT
  -m, --test-time-multiplier FLOAT
  -b, --test-time-base FLOAT
  -s, --swallow-output  

## Run `mutmut` and interprete the results

In [26]:
!mutmut run --paths-to-mutate tests_shipping_fee/calculate_shipping_fee.py --tests-dir tests_shipping_fee


- Mutation testing starting -

These are the steps:
1. A full test suite run will be made to make sure we
   can run the tests successfully and we know how long
   it takes (to detect infinite loops for example)
2. Mutants will be generated and checked

Results are stored in .mutmut-cache.
Print found mutants with `mutmut results`.

Legend for output:
🎉 Killed mutants.   The goal is for everything to end up in this bucket.
⏰ Timeout.          Test suite took 10 times as long as the baseline so were killed.
🤔 Suspicious.       Tests took a long time, but not long enough to be fatal.
🙁 Survived.         This means your tests need to be expanded.
🔇 Skipped.          Skipped.

mutmut cache is out of date, clearing it...
1. Running tests without mutations
⠋ Running...Done

2. Checking mutants
⠧ 47/47  🎉 18  ⏰ 0  🤔 0  🙁 29  🔇 0


In [27]:
!mutmut results

To apply a mutant on disk:
    mutmut apply <id>

To show a mutant:
    mutmut show <id>


Survived 🙁 (29)

---- tests_shipping_fee/calculate_shipping_fee.py (29) ----

3-5, 8-14, 18-27, 31-36, 39, 43, 46


In [28]:
!mutmut show -h

Usage: mutmut show [OPTIONS] [ID_OR_FILE]

  Show a mutation diff.

Options:
  --dict-synonyms TEXT
  -h, --help            Show this message and exit.


In [29]:
!mutmut show 1

--- tests_shipping_fee/calculate_shipping_fee.py
+++ tests_shipping_fee/calculate_shipping_fee.py
@@ -10,7 +10,7 @@
     coupon_type: str,
 ) -> int:
     """Calculate shipping fee based on order and delivery conditions."""
-    fee = 0
+    fee = 1
 
     # 1. base fee
     if order_total < 40000:



In [30]:
!for i in $(seq 1 47); do \
    echo "============================"; \
    echo "mutant $i:"; \
    mutmut show $i 2>/dev/null | grep -E '^[+-][^+-]' || echo "(no diff or invalid id)"; \
    echo ""; \
done

mutant 1:
-    fee = 0
+    fee = 1

mutant 2:
-    fee = 0
+    fee = None

mutant 3:
-    if order_total < 40000:
+    if order_total <= 40000:

mutant 4:
-    if order_total < 40000:
+    if order_total < 40001:

mutant 5:
-        fee += 3000
+        fee = 3000

mutant 6:
-        fee += 3000
+        fee -= 3000

mutant 7:
-        fee += 3000
+        fee += 3001

mutant 8:
-    if weight_kg <= 5:
+    if weight_kg < 5:

mutant 9:
-    if weight_kg <= 5:
+    if weight_kg <= 6:

mutant 10:
-        fee += 0
+        fee = 0

mutant 11:
-        fee += 0
+        fee -= 0

mutant 12:
-        fee += 0
+        fee += 1

mutant 13:
-    elif weight_kg <= 20:
+    elif weight_kg < 20:

mutant 14:
-    elif weight_kg <= 20:
+    elif weight_kg <= 21:

mutant 15:
-        fee += 2000
+        fee = 2000

mutant 16:
-        fee += 2000
+        fee -= 2000

mutant 17:
-        fee += 2000
+        fee += 2001

mutant 18:
-        fee += 5000
+        fee = 5000

mutant 19:
-        f

### ‼️ Before running another test, remove the cache.

In [31]:
!rm -rf .mutmut-cache

### Run `mutmut` with `disable/enagle-mutation-types` option.

In [32]:
!mutmut run --disable-mutation-types XXX --paths-to-mutate tests_shipping_fee/calculate_shipping_fee.py --tests-dir tests_shipping_fee

Usage: mutmut run [OPTIONS] [ARGUMENT]
Try 'mutmut run -h' for help.

Error: The following are not valid mutation types: XXX. Valid mutation types are: operator, keyword, number, name, string, fstring, argument, or_test, and_test, lambdef, expr_stmt, decorator, annassign


| Mutation Type | Description |
|---------------|------------|
| **operator** | Modifies arithmetic or comparison operators (e.g., `+` → `-`, `<` → `<=`). |
| **keyword** | Alters Python keywords that affect control flow (e.g., `break`, `continue`, `return`). |
| **number** | Changes numeric literals (e.g., `0` → `1`, `5` → `6`). |
| **name** | Replaces variable names or special constants such as `None`, `True`, or `False`. |
| **string** | Mutates string literals (e.g., modifies or injects characters). |
| **fstring** | Alters formatted string literals (f-strings). |
| **argument** | Modifies function call arguments (e.g., removing or altering arguments). |
| **or_test** | Changes logical `or` expressions (e.g., altering or simplifying `or` conditions). |
| **and_test** | Changes logical `and` expressions (e.g., altering or simplifying `and` conditions). |
| **lambdef** | Mutates lambda function definitions. |
| **expr_stmt** | Modifies standalone expression statements. |
| **decorator** | Alters or removes decorators applied to functions or classes. |
| **annassign** | Mutates annotated assignments (variables with type annotations). |

---

**Source:**  
Mutation type names are derived from the `mutmut` mutation engine implementation (see `node_mutation.py` in the official mutmut repository):  
https://github.com/boxed/mutmut

In [33]:
!mutmut run --disable-mutation-types string,name --paths-to-mutate tests_shipping_fee/calculate_shipping_fee.py --tests-dir tests_shipping_fee 


- Mutation testing starting -

These are the steps:
1. A full test suite run will be made to make sure we
   can run the tests successfully and we know how long
   it takes (to detect infinite loops for example)
2. Mutants will be generated and checked

Results are stored in .mutmut-cache.
Print found mutants with `mutmut results`.

Legend for output:
🎉 Killed mutants.   The goal is for everything to end up in this bucket.
⏰ Timeout.          Test suite took 10 times as long as the baseline so were killed.
🤔 Suspicious.       Tests took a long time, but not long enough to be fatal.
🙁 Survived.         This means your tests need to be expanded.
🔇 Skipped.          Skipped.

mutmut cache is out of date, clearing it...
1. Running tests without mutations
⠋ Running...Done

2. Checking mutants
⠸ 45/45  🎉 17  ⏰ 0  🤔 0  🙁 28  🔇 0


In [35]:
!mutmut results

To apply a mutant on disk:
    mutmut apply <id>

To show a mutant:
    mutmut show <id>


Survived 🙁 (28)

---- tests_shipping_fee/calculate_shipping_fee.py (28) ----

3-5, 8-14, 18-27, 31-36, 38, 44


In [36]:
!for i in $(seq 1 45); do \
    echo "============================"; \
    echo "mutant $i:"; \
    mutmut show $i 2>/dev/null | grep -E '^[+-][^+-]' || echo "(no diff or invalid id)"; \
    echo ""; \
done

mutant 1:
-    fee = 0
+    fee = 1

mutant 2:
-    fee = 0
+    fee = None

mutant 3:
-    if order_total < 40000:
+    if order_total <= 40000:

mutant 4:
-    if order_total < 40000:
+    if order_total < 40001:

mutant 5:
-        fee += 3000
+        fee = 3000

mutant 6:
-        fee += 3000
+        fee -= 3000

mutant 7:
-        fee += 3000
+        fee += 3001

mutant 8:
-    if weight_kg <= 5:
+    if weight_kg < 5:

mutant 9:
-    if weight_kg <= 5:
+    if weight_kg <= 6:

mutant 10:
-        fee += 0
+        fee = 0

mutant 11:
-        fee += 0
+        fee -= 0

mutant 12:
-        fee += 0
+        fee += 1

mutant 13:
-    elif weight_kg <= 20:
+    elif weight_kg < 20:

mutant 14:
-    elif weight_kg <= 20:
+    elif weight_kg <= 21:

mutant 15:
-        fee += 2000
+        fee = 2000

mutant 16:
-        fee += 2000
+        fee -= 2000

mutant 17:
-        fee += 2000
+        fee += 2001

mutant 18:
-        fee += 5000
+        fee = 5000

mutant 19:
-        f

## 🎯 Exercise: Kill All Mutants

In this exercise, your goal is to design a **strong test suite** that kills all generated mutants of `calculate_shipping_fee.py`.

You are given an initial set of tests in `tests_shipping_fee/test_calculate_shipping_fee.py`.  
However, the current tests may not be sufficient — some mutants may survive.

### Your Task

1. Run mutation testing using `mutmut` (with `--paths-to-mutate tests_shipping_fee/calculate_shipping_fee.py` and `--tests-dir tests_shipping_fee`).
2. Identify the survived mutants using: `mutmut results/show`
3. Analyze why each survived mutant was not detected.
4. Improve `tests_shipping_fee/test_calculate_shipping_fee.py` by adding meaningful test cases.
5. Re-run mutation testing and repeat until **all mutants are killed**.